Note: Run this notebook in Google Colab


This notebook is essentially an end-to-end NLP pipeline using Hugging Face Transformers on Dhurandhar 2 movie reviews.

Key NLP Tasks Demonstrated
* Sentiment Analysis → Positive/Negative classification
* Machine Translation → English → Spanish
* Question Answering → Extract information from reviews
* Text Summarization → Generate concise review summaries
* Model Evaluation → Accuracy, F1 Score, BLEU Score

In [ ]:
# pip install transformers evaluate

In [ ]:
import transformers
print(transformers.__version__)

5.10.1


In [ ]:
# !pip install -U transformers sentencepiece sacremoses
# !pip install evaluate

In [ ]:
import pandas as pd
import torch

# Load the car reviews dataset
file_path = "/content/sample_data/netflix movie dhurandhar 2.csv"
df = pd.read_csv(file_path, delimiter=";")

# Put the car reviews and their associated sentiment labels in two lists
reviews = df['Review'].tolist()
real_labels = df['Class'].tolist()


# Instruction 1: sentiment classification


# Load a sentiment analysis LLM into a pipeline
from transformers import pipeline
classifier = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

# The Pipeline is a simple but powerful inference API that is readily
# available for a variety of machine learning tasks with any model from the Hugging Face Hub.

# This pipeline performs text classification on a given input and determines if its a
# positive or negative sentiment. You can pass
# multiple texts to the same pipeline, which will be processed and passed through the model together as a batch.

# distilbert-base-uncased-finetuned-sst-2-english: This model is a fine-tune checkpoint of DistilBERT-base-uncased,
# fine-tuned on SST-2.

# Developed by: Hugging Face
# Model Type: Text Classification

# This model can be used for topic classification. You can use the raw model for either masked language
# modeling or next sentence prediction, but it's mostly intended to be fine-tuned on a downstream task.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
print(reviews)

["Aditya Dhar delivers a gripping and ambitious crime-thriller with Dhurandhar: The Revenge. From the very first scene, the film pulls you into a dangerous world of espionage, loyalty, betrayal, and survival. The story follows Jaskirat Singh Rangi as he fully embraces his identity as Hamza Ali Mazari and rises through Karachi's criminal underworld, creating a tense and emotionally charged journey. Ranveer Singh is absolutely phenomenal in the lead role. His transformation is intense, convincing, and filled with emotional depth. He brings both vulnerability and ruthless determination to the character, making Hamza Ali Mazari one of the most memorable protagonists in recent Indian cinema. Industry veterans have even praised his performance as award-worthy. The supporting cast is equally impressive. Akshaye Khanna delivers a powerful and nuanced performance, while Sanjay Dutt commands every scene with his presence. The ensemble cast elevates the narrative and keeps the tension high throug

In [ ]:
print(real_labels)

['POSITIVE', 'NEGATIVE', 'POSITIVE', 'NEGATIVE', 'POSITIVE']


In [ ]:
reviews_example="The movie was fantastic!"
predicted_labels1 = classifier(reviews_example)
print(predicted_labels1 )


[{'label': 'POSITIVE', 'score': 0.9998776912689209}]


In [ ]:
# Perform inference on the car reviews and display prediction results
predicted_labels = classifier(reviews)
for review, prediction, label in zip(reviews, predicted_labels, real_labels):
    print(f"Review: {review}\nActual Sentiment: {label}\nPredicted Sentiment: {prediction['label']} (Confidence: {prediction['score']:.4f})\n")

# Load accuracy and F1 score metrics
import evaluate
# Evaluate is a library that makes evaluating
# and comparing models and reporting their performance easier and more standardized.
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

Review: Aditya Dhar delivers a gripping and ambitious crime-thriller with Dhurandhar: The Revenge. From the very first scene, the film pulls you into a dangerous world of espionage, loyalty, betrayal, and survival. The story follows Jaskirat Singh Rangi as he fully embraces his identity as Hamza Ali Mazari and rises through Karachi's criminal underworld, creating a tense and emotionally charged journey. Ranveer Singh is absolutely phenomenal in the lead role. His transformation is intense, convincing, and filled with emotional depth. He brings both vulnerability and ruthless determination to the character, making Hamza Ali Mazari one of the most memorable protagonists in recent Indian cinema. Industry veterans have even praised his performance as award-worthy. The supporting cast is equally impressive. Akshaye Khanna delivers a powerful and nuanced performance, while Sanjay Dutt commands every scene with his presence. The ensemble cast elevates the narrative and keeps the tension high 

In [ ]:

# Map categorical sentiment labels into integer labels
references = [1 if label == "POSITIVE" else 0 for label in real_labels]
predictions = [1 if label['label'] == "POSITIVE" else 0 for label in predicted_labels]

# Calculate accuracy and F1 score
accuracy_result_dict = accuracy.compute(references=references, predictions=predictions)
accuracy_result = accuracy_result_dict['accuracy']
f1_result_dict = f1.compute(references=references, predictions=predictions)
f1_result = f1_result_dict['f1']
print(f"Accuracy: {accuracy_result}")
print(f"F1 result: {f1_result}")

Accuracy: 1.0
F1 result: 1.0


Translate Review from English to Spanish

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# AutoTokenizer: This is a generic tokenizer class that will be instantiated as
# one of the tokenizer classes of the
# library when created with the AutoTokenizer.from_pretrained() class method.
AutoTokenizer: simply converts the words into tokens
 ( A numerical representation of text) however this alone doesn't produce sentence embeddings

# AutoModelForSeq2SeqLM:  is a generic model class that will be instantiated as
# one of the model classes of the library (with a sequence-to-sequence language modeling head)
# when created with the from_pretrained() class method or the from_config() class method.
# I is used when both your input and output are text,
# but the output is different from the input. It is a pretrained encoder–decoder
# model designed for text-to-text generation.

# Use AutoModelForSeq2SeqLM if your task is:

# Summarization
# Translation
# Paraphrasing
# Question generation
# Grammar correction

model_name = "Helsinki-NLP/opus-mt-en-es"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# AutoTokenizer: What it does:

# Splits text into tokens (subwords or characters)
# Maps tokens to integer IDs
# Handles padding, truncation, attention masks, and special tokens

# Used in:

# NLP (text, QA, summarization, translation)
# Multimodal models (text + image/audio)
# Training and inference pipelines

# What AutoTokenizer Handles

# Subword tokenization (WordPiece, BPE, Unigram)
# Input IDs
# Attention masks
# Token type IDs (for sentence pairs)
# Padding & truncation
# Special tokens ([CLS], [SEP], <s>, etc.)


model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

first_review = reviews[0]

inputs = tokenizer(
    first_review,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

translated_tokens = model.generate(**inputs)

translated_review = tokenizer.decode(
    translated_tokens[0],
    skip_special_tokens=True
)

print("Model translation:")
print(translated_review)

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model translation:
Aditya Dhar ofrece un apasionante y ambicioso juego del crimen con Dhurandhar: The Revenge. Desde la primera escena, la película te lleva a un peligroso mundo de espionaje, lealtad, traición y supervivencia. La historia sigue a Jaskirat Singh Rangi mientras abraza plenamente su identidad como Hamza Ali Mazari y se eleva a través del inframundo criminal de Karachi, creando un viaje tenso y emocionalmente cargado. Ranveer Singh es absolutamente fenomenal en el papel principal. Su transformación es intensa, convincente y llena de profundidad emocional. Él aporta vulnerabilidad y determinación sin cuartel al personaje, haciendo que Hamza Ali Mazari sea uno de los protagonistas más memorables de su reciente cine indio. Los veteranos de la industria incluso han elogiado su actuación como merecedor de premios. El reparto de apoyo es igualmente impresionante. Akshaye Khanna ofrece una actuación poderosa y matizada, mientras que Sanjay Dutt dirige todas las escenas con su pre

Evaluate Translation using (BLEU Score)

In [ ]:
import evaluate

with open("/content/sample_data/reference_translations.txt", "r") as file:
    references = [line.strip() for line in file.readlines()]

bleu = evaluate.load("bleu")

# BLEU (Bilingual Evaluation Understudy) is a measurement of the difference
# between an automatic translation and human-created reference translations of the same source sentence.

# A BLEU score is a number between zero and 100. A score of zero indicates a low-quality translation where
# nothing in the translation matched the reference. A score of 100 indicates a perfect translation
# that's identical to the reference. It's
# not necessary to attain a score of 100—a BLEU score between 40 and 60 indicates a high-quality translation.



bleu_score = bleu.compute(
    predictions=[translated_review],
    references=[references]
)

print("BLEU:", bleu_score["bleu"])

BLEU: 0.0


In [ ]:

# Instruction 3: extractive QA

# Import auto classes (optional: can be solved via pipelines too)
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering

# Instantiate model and tokenizer
model_ckp = "deepset/minilm-uncased-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_ckp)
model = AutoModelForQuestionAnswering.from_pretrained(model_ckp)

# Define context and question, and tokenize them
context = reviews[1]
print(f"Context:\n{context}")
question = "What is the best opinion of the movie?"
inputs = tokenizer(question, context, return_tensors="pt")

# Perform inference and extract answer from raw outputs
with torch.no_grad():
  outputs = model(**inputs)
start_idx = torch.argmax(outputs.start_logits)
end_idx = torch.argmax(outputs.end_logits) + 1
answer_span = inputs["input_ids"][0][start_idx:end_idx]

# Decode and show answer
answer = tokenizer.decode(answer_span)
print("Answer: ", answer)





Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: deepset/minilm-uncased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Context:
After the success of the first film, I expected Dhurandhar 2 to raise the stakes and deliver a gripping continuation of Jaskirat Singh Rangi's journey into Karachi's criminal underworld. Unfortunately, the film ends up being a bloated and uneven experience that struggles to justify its lengthy runtime. The biggest issue is the screenplay. Despite an interesting premise involving undercover operations, gang rivalries, loyalty, and betrayal, the narrative moves at a painfully slow pace. Large portions of the film feel repetitive, with the same themes and conflicts being revisited without adding anything meaningful to the story. What should have been a tense crime-thriller often turns into a drawn-out exercise in patience. Ranveer Singh puts in effort as Hamza Ali Mazari, but the character is written almost invincible, leaving little room for suspense or emotional investment. The film repeatedly tells us how dangerous and intelligent he is instead of showing genuine challenges th

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "cnicu/t5-small-booksum"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text_to_summarize = reviews[-1]

# T5 models usually expect a "summarize:" prefix
input_text = "summarize: " + text_to_summarize

inputs = tokenizer(
    input_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

summary_ids = model.generate(
    **inputs,
    max_length=53,
    min_length=20,
    num_beams=4,
    early_stopping=True
)

summarized_text = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Summarized text:")
print(summarized_text)

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Summarized text:
Aditya Dhar crafts a dark and immersive world where loyalty, betrayal, ambition, and survival collide in spectacular fashion. The film's premise revolves around an undercover operative navigating Pakistan's criminal networks while
